In [6]:
import numpy as np
import pandas as pd
import sys
sys.path.append('../src')
from heatexchanger_model import effectiveness_ntu

#This creates our time line (1 reading per minute for 4 hours)
timestamps = pd.date_range(start="2026-01-01 00:00", periods=240, freq="1min") #New syntax: pd.date_range() generates a sequence of timestamps automatically

#Generate operating conditions around temp 80 randomly within 0.5 at the heat inlet
Th_i_values = np.random.normal(loc=80, scale=0.5, size=240) #New syntax: np.random.normal() generates random normally distributed values around your target

#Generate operating conditions around temp 20 randomly within 0.5 at the cold inlet
Tc_i_values = np.random.normal(loc=20, scale=0.5, size=240)

#Generate operating conditions for the mass flow in at the heat inlet within 0.03 kg/s 
mdot_h_values = np.random.normal(loc=1.5, scale=0.03, size=240)

#Generate operating conditions for the mass flow in at the cold inlet within 0.02 kg/s
mdot_c_values = np.random.normal(loc=1.0, scale=0.02, size=240)

cp = 4180 #specific heat of water in kj/molk
U = 850 #Overall heat transfer coefficient in W/m^2k
A = 12 #Heat transfer area in m^2

results_list = [] #Creates a list to append our simulated data values to

#Loop to create 4 hours of data
for i in range(240):
    results = effectiveness_ntu(Th_i_values[i], Tc_i_values[i], mdot_h_values[i], mdot_c_values[i], cp, U, A)
    results_list.append(results)

#Creating sensor noise for both hot and cold outlets
sensor_noise_Th = np.random.normal(loc=0, scale=0.2, size=240)
sensor_noise_Tc = np.random.normal(loc=0, scale=0.2, size=240)

#Extract the Th_o results
Th_o_values = np.array([r['Th_o'] for r in results_list])
Tc_o_values = np.array([r['Tc_o'] for r in results_list])

#Final Th and Tc values
Th_o_measured = Th_o_values + sensor_noise_Th
Tc_o_measured = Tc_o_values + sensor_noise_Tc

#pandas dataframe / creating dictionary
dataframe = pd.DataFrame({"Timestamp": timestamps, "Th_i": Th_i_values, "Tc_i": Tc_i_values, "mdot_h": mdot_h_values, "mdot_c": mdot_c_values, "Th_o_true": Th_o_values, "Tc_o_true": Tc_o_values, "Th_o_measured": Th_o_measured, "Tc_o_measured": Tc_o_measured})

#Save the dataframe to CSV file
dataframe.to_csv("../data/baseline_data.csv", index=False)

,Timestamp,Th_i,Tc_i,mdot_h,mdot_c,Th_o_true,Tc_o_true,Th_o_measured,Tc_o_measured
0,2026-01-01 00:00:00,80.058821,20.320897,1.501618,1.006638,53.465577,59.990469,53.821968,60.037953
1,2026-01-01 00:01:00,79.549271,20.011975,1.526038,1.003121,53.386804,59.812656,53.235173,60.078704
2,2026-01-01 00:02:00,80.688575,20.134730,1.476073,1.015815,53.282348,59.958516,53.448775,59.861248
3,2026-01-01 00:03:00,79.806445,21.272723,1.547286,1.002655,54.335906,60.578565,54.178603,60.475936
4,2026-01-01 00:04:00,80.622614,20.737715,1.446342,0.989114,53.535197,60.346576,53.351698,60.433018
